--------
Create spark session
--------
--------

In [8]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os

# Load env
load_dotenv()

mongo_uri = os.getenv("MONGO_URI")
mongo_db = os.getenv("MONGO_DB")

# Combine into full URI
mongo_connection = f"{mongo_uri}{mongo_db}"

# Minio credentials
minio_endpoint  = os.getenv("MINIO_ENDPOINT") 
minio_access_key = os.getenv("MINIO_ACCESS_KEY")
minio_secret_key = os.getenv("MINIO_SECRET_KEY")

spark = SparkSession.builder \
    .appName("Olist Transformation") \
    .config("spark.jars.packages",
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.367") \
    \
    .config("spark.mongodb.read.connection.uri", mongo_connection) \
    .config("spark.mongodb.write.connection.uri", mongo_connection) \
    \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{minio_endpoint}") \
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

--------
Read data from Mongodb
--------
--------

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("price", DoubleType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_value", DoubleType(), True),
])

In [10]:
# Read collections
orders = spark.read.format("mongodb") \
    .option("collection", "orders") \
    .schema(orders_schema) \
    .load()

order_items = spark.read.format("mongodb") \
    .option("collection", "order_items") \
    .schema(order_items_schema) \
    .load()

products = spark.read.format("mongodb") \
    .option("collection", "products") \
    .schema(products_schema) \
    .load()

customers = spark.read.format("mongodb") \
    .option("collection", "customers") \
    .schema(customers_schema) \
    .load()

payments = spark.read.format("mongodb") \
    .option("collection", "payments") \
    .schema(payments_schema) \
    .load()

In [11]:
orders.printSchema()
order_items.printSchema()
products.printSchema()
customers.printSchema()
payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_value: double (nullable = true)



In [12]:
try:
    orders.limit(5).show()
except Exception as e:
    print(str(e))

+--------------------+--------------------+------------------------+
|            order_id|         customer_id|order_purchase_timestamp|
+--------------------+--------------------+------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|     2017-10-02 10:56:33|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|     2018-07-24 20:41:37|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|     2018-08-08 08:38:49|
|949d5b44dbf5de918...|f88197465ea7920ad...|     2017-11-18 19:28:06|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|     2018-02-13 21:18:39|
+--------------------+--------------------+------------------------+



--------
Transform
--------
--------

In [13]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.functions import sum

# Select columns
orders = orders.select("order_id", "customer_id", "order_purchase_timestamp")
order_items = order_items.select("order_id", "product_id", "price")
products = products.select("product_id", "product_category_name")
customers = customers.select("customer_id", "customer_city", "customer_state")
payments = payments.select("order_id", "payment_value")

# Aggregate payment
payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("payment_value"))

--------
Join tables and load to silver Minio
--------
--------

In [14]:
# Transform
df_clean = orders \
    .join(order_items, "order_id") \
    .join(products, "product_id") \
    .join(customers, "customer_id") \
    .join(payments_agg, "order_id") \
    .withColumn("order_date", to_timestamp(col("order_purchase_timestamp"))) \
    .drop("order_purchase_timestamp")

# Save to MinIO (Silver Layer)
df_clean.write \
    .mode("overwrite") \
    .parquet("s3a://olist-data/silver/fact_orders")

print("Silver data saved to MinIO")

26/04/02 20:41:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Silver data saved to MinIO
